In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.4 Embeddings

> 🎯 **Goal:** Turn each token id into a learned vector of numbers the model can actually compute with.

**Why this matters.** Tokenization gave us integers, but an integer carries no meaning. Token id 42 is not "bigger" or "more like" id 43 in any useful sense; the numbers are just labels. The embedding layer is where those bare labels become rich vectors that the rest of the network reshapes into meaning. Every transformer begins with this lookup.

**The intuition.** Picture a dictionary. You look up a word (the id) and get back its definition (a vector of numbers that captures what the token is like). The embedding table is that dictionary: one entry per token, each entry a list of numbers. At first the definitions are random gibberish; training slowly rewrites them so that tokens used in similar ways end up with similar vectors.

**The mechanics.** An *embedding table* is just a matrix of shape `(vocab_size, dim)`. *Dim* (the embedding dimension) is how many numbers describe each token. Row `i` is the vector for token id `i`. `nn.Embedding(vocab, dim)` creates this table; calling `emb(ids)` is a pure *lookup*: it gathers the rows named by the ids, so a list of 3 ids returns 3 vectors of length `dim`. Crucially the table is *learned*: those numbers are parameters, and gradient descent during training nudges them to be useful.

In [ ]:
from torch import nn

vocab, dim = 50, 8
emb = nn.Embedding(vocab, dim)
ids = torch.tensor([1, 2, 3])
vecs = emb(ids)
print("ids", ids.tolist(), "-> vectors of shape", tuple(vecs.shape))

**What just happened.** Three ids went in and came out as a tensor of shape `(3, 8)`: three vectors, each with 8 numbers (`dim`). The print confirms the lookup mapped each id to its own row. The assertion `emb(torch.tensor([1]))[0] == emb.weight[1]` proves the operation really is row selection: feeding id 1 returns exactly row 1 of the weight matrix, nothing more.

A key consequence: the same id always returns the same row. So two copies of the token "the" start life as identical vectors. They only become different later, when attention layers mix in surrounding context. The embedding gives identity; the network gives nuance.

⚠️ **Common pitfalls**
- Id out of range: an id of `vocab` or higher (here 50 or more) indexes past the table and raises an error. Your tokenizer's vocab size and the embedding's `vocab` argument must match exactly.
- Expecting meaning before training: a freshly created embedding is random. Similar tokens do not have similar vectors yet. That structure is learned over many training steps; it is not there at initialization.

🏋️ **Try it yourself.** Encode the same id twice, like `torch.tensor([2, 2])`, and check the two output rows are identical. Then bump `dim` from 8 to 32 and rerun: the shape becomes `(3, 32)`, showing `dim` is a free knob you choose for model capacity.

In [ ]:
assert tuple(vecs.shape) == (3, 8)
assert torch.equal(emb(torch.tensor([1]))[0], emb.weight[1])